In [1]:
import pandas as pd

df_answers=pd.read_csv('data/rag-answers-new.csv')

answers=df_answers.to_dict(orient="records")

In [2]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str=Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [3]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [4]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [6]:
test = answers[10]

In [8]:
prompt = aqa_judge_prompt.format(
    question=test["question"],
    answer_orig=test["answer_orig"],
    answer_llm=test["answer_llm"]
)

In [9]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer conveys the same core information as the ground truth: students do not use the Zoom link, which is only for instructors/presenters/TAs; they join via YouTube Live; the stream URL is posted in Telegram/Slack announcements and on the DataTalksClub YouTube channel; and questions go to Slido. It omits the caution about not posting questions in chat, but that is a minor detail and does not change the main answer.', score='good')

In [15]:
calc_price(usage)

{'input_cost': 0.0003285,
 'output_cost': 0.0004905,
 'total_cost': 0.0008190000000000001}

In [16]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [18]:
eval_result, usage= evaluate_aqa(
    question=test["question"],
    answer_orig=test["answer_orig"],
    answer_llm=test["answer_llm"]
)

In [19]:
eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says students join via YouTube Live, the Zoom link is only for instructors/presenters/TAs, the stream URL is posted in Telegram/Slack announcements, YouTube channel is available, and questions go to Slido. It preserves the key point that students should not use Zoom directly.', score='good')

In [20]:
calc_price(usage)

{'input_cost': 0.0003285,
 'output_cost': 0.00037349999999999997,
 'total_cost': 0.0007019999999999999}

All answers

In [21]:
def judge_record(que):
    eval_result, usage = evaluate_aqa(
        question=que["question"],
        answer_orig=que["answer_orig"],
        answer_llm=que["answer_llm"]
    )

    result = {
        "question": que["question"],
        "document": que["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [22]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/315 [00:00<?, ?it/s]

In [23]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [24]:
calc_total_price(usages)

0.20567024999999992

In [25]:
df_eval = pd.DataFrame(evaluations)

In [33]:
df_eval.loc[:15]

,question,document,score,reasoning
0,Is it okay to join the course late if I just f...,74eb249bbf,good,The AI answer preserves the ground truth meani...
1,Can I still take this course even if I missed ...,74eb249bbf,bad,The AI answer captures the first part: you can...
2,If I join after the course has already started...,74eb249bbf,good,The AI answer preserves the key point: late jo...
3,Do I need to submit my project before submissi...,74eb249bbf,good,The AI answer preserves the key point exactly:...
4,I’m a bit late to the course—what do I need to...,74eb249bbf,good,"The ground truth says that if you're late, you..."
5,I registered for the LLM Zoomcamp — when shoul...,977bf7786c,good,The AI answer conveys the same core points as ...
6,Do I actually need an acceptance email before ...,977bf7786c,good,The AI answer matches the ground truth: it sta...
7,"If I filled out the registration form, does th...",977bf7786c,good,The AI answer matches the ground truth: it say...
8,"Is registering for the LLM Zoomcamp required, ...",977bf7786c,good,The AI answer matches the ground truth: regist...
9,Can I begin learning and submit homework even ...,977bf7786c,good,The AI answer preserves the key point: no conf...


In [31]:
df_eval.score.value_counts()

score
good    290
bad      25
Name: count, dtype: int64

In [34]:
df_eval.score.value_counts(normalize=True)

score
good    0.920635
bad     0.079365
Name: proportion, dtype: float64

In [35]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)